## Chapter2 : Working with Text data

In [ ]:
from importlib.metadata import version
# The importlib module in Python provides the implementation of the import statement and the __import__() function, offering a programmatic way to interact with Python's import system. It exposes components that allow for dynamic module loading, customization of the import process, and access to package metadata.

print("Torch version: ", version("torch"))
print("Torchvision version: ", version("torchvision"))


Torch version:  2.9.0+cu126
Torchvision version:  0.24.0+cu126


This chapter covers the data preperation and sampling to get input data "ready" for LLM.

### Understanding Word Embedding
**There are many forms of embeddings, such as * Video Embeddings, * Audio Embeddings and * text embeddings , we focus on the text embeddings here...

In [ ]:
# Tokenizing the text : Breaking the text into smaller units
# such as individual words, and punctuation characters

import os
import requests

if not os.path.exists("the-verdict.txt"):
  url = (
      "https://raw.githubusercontent.com/rasbt/"
      "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
      "the-verdict.txt"
)

  file_path = "the-verdict.txt"

  response = requests.get(url, timeout=30)
  response.raise_for_status()
  with open(file_path, "wb") as f:
    f.write(response.content)






In [ ]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [ ]:
import re
# THis will split the text into whitespaces


text = "Hello, world, This is a test"
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world,', ' ', 'This', ' ', 'is', ' ', 'a', ' ', 'test']


Here we split the text on whitespaces but we also want commas and periods , so we modify the regex: to do that as well.....

In [ ]:
result = re.split(r'[,.]||\s', text)

print(result)

['', 'H', 'e', 'l', 'l', 'o', '', '', '', 'w', 'o', 'r', 'l', 'd', '', '', '', 'T', 'h', 'i', 's', '', '', 'i', 's', '', '', 'a', '', '', 't', 'e', 's', 't', '']


And as we can see it make empty strings, so we remove them ....

In [ ]:
# strip whitespace from each item and then filter out any empty string,,,,,

result = [item for item in result if item.strip()]
print(result)

['H', 'e', 'l', 'l', 'o', 'w', 'o', 'r', 'l', 'd', 'T', 'h', 'i', 's', 'i', 's', 'a', 't', 'e', 's', 't']



* The above is good but we need to add other types of punctuations, such as periods, questions marks and so on..

In [ ]:
text = "Hello , world, Is this -- a test?"

result = re.split(r'([,.:;?_!"()\']|--|\s)', text)
result = [item.strip() for item in result if item.strip()]

print(result)

['Hello', ',', 'world', ',', 'Is', 'this', '--', 'a', 'test', '?']


In [ ]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed=[item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


* Calculating the total number of tokens


In [ ]:
print(len(preprocessed))

4690


## Converting token into token ids...

* Next, we convert the text tokens into token IDs that we can process via embedding layers later
* From these tokens, we can now build a vocabulary that consists of all the uniwue tokens



In [ ]:
all_words = sorted(set(preprocessed))

vocab_size = len(all_words)

print(vocab_size)

1130


In [ ]:
vocab = {token:integer for integer , token in enumerate(all_words)}

* The last 50 entries in this vocabulary


In [ ]:
# for i, item in enumerate(list(vocab.items())[:50]):
#     print(item)
for i, item in enumerate(list(vocab.items())):
    print(item)

    if i >= 50:
      break


('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


* Here we put all these into a tokenizer class ...

In [25]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text


The `encode` function tunrs the text into token IDs
and `decode` function tunrs the token back into text

* We can use the tokenizer to encode (that is, tokenize) texts into integers
* These integers can then be embedded (later) as input of/for the LLM

In [27]:
tokenizer = SimpleTokenizerV1(vocab)

text = """ "It's the last he painted, you know."
  Mrs. Gisburn said with pardonable pride. """

ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 7, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


* We can decode the integers back into text


In [28]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know." Mrs. Gisburn said with pardonable pride.'

In [30]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know." Mrs. Gisburn said with pardonable pride.'

### Adding special  context tokens ...

* It's useful to add some "special" tokens for unknown words and to denote the end of a text


* Some tokenizers use special tokens to help the LLM  with additional context..


* SOme od these special tokens are:
`[BOS]` (beginning of the sequence) marks the beginning of text.
`[EOS]` (end of sequence) marks where the text ends ( this is usually used to cancatenate multiple unrealted texts, two different wikipedia articles or 2 different books and so on)

`[PAD]` (padding ) if we train LLMs with a batch size greater than 1 (we may include multiple texts with different lengths; with the padding token we pad the shorter texts to the longest length so that all texts have an equal length

`[UNK]` :to represent words that are not included in the vocabulary.


Note that GPT-2 does not need any of these tokens mentioned above but only uses an `<|endoftext|>` token to reduce complexity


The `<|endoftext|>` is analogous to the [EOS] token mentioned above.

GPT also uses the `<|endoftext|>` for padding (since we typically use a mask when training on batched inputs, we would not attend padded tokens anyways, so it does not matter what these tokens are)

GPT-2 does not use an `<UNK>` token for out-of-vocabulary words; instead, GPT-2 uses a byte-pair encoding `(BPE)` tokenizer, which breaks down words into subword units which we will discuss in a later section.



  We use the `<|endoftext|>`tokens between two independent sources of text:





In [31]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)


KeyError: 'Hello'

* The above produces an error because the word "Hello" is not contained in the vocabulary

* To deal with such cases, we can add special tokens like "<|unk|>" to the vocabulary to represent unknown words

* Since we are already extending the vocabulary, let's add another token called `"<|endoftext|>"` which is used in GPT-2 training to denote the end of a text (and it's also used between concatenated text, like if our training datasets consists of multiple articles, books, etc.)


In [34]:
all_tokens = sorted(list(set(preprocessed)))

all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer , token in enumerate (all_tokens)}

In [35]:
len(vocab.items())

1132

In [37]:
for i, item in enumerate(list(vocab.items())[-5:]):
  print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


* We also need to adjust the tokenizer accordingly so that it knows when and how to use the new `<unk>` token...

In [62]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text


Let tokenize text with modified tokenizer


In [67]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [69]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [72]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

## BytePair Encoding ....

* GPT-2 used BytePair encoding (BPE) as its tokenizer

* it allows the model to break down words that aren't in its predefined vocabulary into smaller subword units or even individual characters, enabling it to handle out-of-vocabulary words

* For instance, if GPT-2's vocabulary doesn't have the word "unfamiliarword," it might tokenize it as ["unfam", "iliar", "word"] or some other subword breakdown, depending on its trained BPE merges

* In this chapter, we are using the BPE tokenizer from OpenAI's open-source tiktoken library, which implements its core algorithms in Rust to improve computational performance





In [74]:
 !pip install tiktoken


In [76]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.12.0


In [77]:
tokenizer = tiktoken.get_encoding("gpt2")

In [81]:
text = (
    "Hello do you like tea? <|endoftext|> In the Sunlit terraces"
    "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 3825, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [82]:
strings = tokenizer.decode(integers)

print(strings)

Hello do you like tea? <|endoftext|> In the Sunlit terracesof someunknownPlace.


## Data Sampling with Sliding window ....

* We train LLMs to generate one word at a time, so we want to prepare the training data accordingly where the next word in a sequence represents the target to predict:



In [83]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
  raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))


5145


* For each text chunk, we want the inputs and targets

* Since we want to predict the next word, the targets are the inputs shifted by one position to the right ...

In [84]:
enc_sample = enc_text[50:]

In [85]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size]

print(f"x: {x}")
print(f"y:             {y}")

x: [290, 4920, 2241, 287]
y:             [4920, 2241, 287]


* one by one the prediction will look like as this ...

In [86]:
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]


  print(context, '----->', desired)

[290] -----> 4920
[290, 4920] -----> 2241
[290, 4920, 2241] -----> 287
[290, 4920, 2241, 287] -----> 257


In [87]:
for i in range(1, context_size+1):
  context = enc_sample[:i]
  desired = enc_sample[i]

  print(tokenizer.decode(context), '------>', tokenizer.decode([desired]))

 and ------>  established
 and established ------>  himself
 and established himself ------>  in
 and established himself in ------>  a


* Next : we implement a simple data laoder that iterate over the input dataset and returns the inputs and target shifted by one.


In [88]:
import torch
print("PyTorch_version", torch.__version__)


PyTorch_version 2.9.0+cu126


* We use SlidingWindow approach , changing position by +1

* Create the dataset and dataloader that extract chunks from input text dataset is the next step .....